# ❤️ Heart Disease Prediction
This notebook aims to predict the presence of heart disease in patients using medical attributes from the Heart Disease UCI dataset.

## 1. Data Preparation
### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Set plot style
sns.set(style='whitegrid')
print("Libraries loaded.")

### Load Dataset

In [ ]:
df = pd.read_csv('heart_disease_uci.csv')
print(f"Dataset shape: {df.shape}")
df.head()

### Exploratory Data Analysis (EDA)
We check for missing values and visualize the target distribution.

In [ ]:
# Target variable transformation: 0 = No disease, 1 = Disease present
df['target'] = df['num'].apply(lambda x: 1 if x > 0 else 0)

plt.figure(figsize=(8, 5))
sns.countplot(x='target', data=df, palette='viridis')
plt.title('Distribution of Heart Disease (0: No, 1: Yes)')
plt.show()

In [ ]:
# Age distribution
plt.figure(figsize=(10, 6))
sns.histplot(df['age'], kde=True, color='red')
plt.title('Age Distribution of Patients')
plt.show()

### Preprocessing
We handle missing values, drop irrelevant columns, and prepare the features.

In [ ]:
# Drop columns that aren't useful for prediction
data = df.drop(['id', 'dataset', 'num'], axis=1)

X = data.drop('target', axis=1)
y = data['target']

# Identify numerical and categorical columns
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'bool']).columns.tolist()

print(f"Numerical features: {numeric_features}")
print(f"Categorical features: {categorical_features}")

In [ ]:
# Define preprocessing for numerical data (Impute + Scale)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Define preprocessing for categorical data (Impute + OneHotEncode)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Bundle preprocessing for all features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline ready.")

## 2. Model Training

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train the Logistic Regression model in a pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
print("Model trained.")

## 3. Model Evaluation

In [ ]:
# Predictions
y_pred = model.predict(X_test)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

In [ ]:
# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Disease', 'Disease'], 
            yticklabels=['No Disease', 'Disease'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

## Summary
- The model was trained using a Logistic Regression classifier after handling missing values and scaling features.
- The preprocessing was automated using a `ColumnTransformer` and `Pipeline` for reproducibility.
- Performance metrics and the confusion matrix indicate how well the model distinguishes between patients with and without heart disease.